# Part 1: Data Preprocessing & Feature Engineering 

## Data Download and Schema Overview

### Exploring Yellow Taxi Data File

In [269]:
import polars as pl
import pathlib as pathlb
from sklearn.preprocessing import StandardScaler, OneHotEncoder

file_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
base_dir = pathlb.Path('data/raw')
file_name = 'yellow_tripdata_2024-01.parquet'
file_path = base_dir/file_name

## Checks if file downloaded if not it downloads it
if not file_path.is_file() :
    base_dir.mkdir(parents=True, exist_ok=True)
    print(f" Error {file_name} not found downloading...\n")
    pl.read_parquet(file_url).write_parquet(file_path)
taxi_df = pl.read_parquet(file_path)

print(f"{"SCHEMA"}\n")
print(f"{"Name":<25}Type")
for col, type in taxi_df.schema.items() :
    print(f"{col:<25}{type}")


taxi_df.head()

 Error yellow_tripdata_2024-01.parquet not found downloading...

SCHEMA

Name                     Type
VendorID                 Int32
tpep_pickup_datetime     Datetime(time_unit='ns', time_zone=None)
tpep_dropoff_datetime    Datetime(time_unit='ns', time_zone=None)
passenger_count          Int64
trip_distance            Float64
RatecodeID               Int64
store_and_fwd_flag       String
PULocationID             Int32
DOLocationID             Int32
payment_type             Int64
fare_amount              Float64
extra                    Float64
mta_tax                  Float64
tip_amount               Float64
tolls_amount             Float64
improvement_surcharge    Float64
total_amount             Float64
congestion_surcharge     Float64
Airport_fee              Float64


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


### Exploring Lookup Zone Data File

In [229]:
file_name ='lookup.csv'
file_path = base_dir / file_name
file_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

if not file_path.is_file():
    base_dir.mkdir(parents=True, exist_ok=True)
    print(f" Error {file_name} not found downloading...\n")
    pl.read_csv(file_url,encoding="utf8-lossy").write_csv(file_path)

lookup_df = pl.read_csv(file_path)

print(f"{"SCHEMA"}\n")
print(f"{"Name":<25}Type")
for col, type in lookup_df.schema.items() :
    print(f"{col:<25}{type}")

print(lookup_df['Borough'].value_counts())

SCHEMA

Name                     Type
LocationID               Int64
Borough                  String
Zone                     String
service_zone             String
shape: (8, 2)
┌───────────────┬───────┐
│ Borough       ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ Queens        ┆ 69    │
│ N/A           ┆ 1     │
│ Staten Island ┆ 20    │
│ EWR           ┆ 1     │
│ Unknown       ┆ 1     │
│ Bronx         ┆ 43    │
│ Brooklyn      ┆ 61    │
│ Manhattan     ┆ 69    │
└───────────────┴───────┘


## Data Cleaning


### Cleaning Taxi Parquet

In [102]:
cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime',  'PULocationID', 'DOLocationID', 'fare_amount']
initial_rows = len(taxi_df)

taxi_df = taxi_df.drop_nulls(subset = cols)
null_rows = initial_rows - len(taxi_df)

taxi_df = taxi_df.filter( 
    (~pl.col("fare_amount").is_nan()) & 
    (~pl.col("trip_distance").is_nan()))
nan_rows = (initial_rows - null_rows) - len(taxi_df)


taxi_df = taxi_df.filter((pl.col('trip_distance') > 0))
distance_rows = (initial_rows - null_rows - nan_rows) - len(taxi_df)

taxi_df = taxi_df.filter((pl.col('fare_amount') > 0)& (pl.col('fare_amount') <= 500))
fare_rows = (initial_rows - null_rows - nan_rows - distance_rows) - len(taxi_df)

taxi_df = taxi_df.filter(pl.col('tpep_dropoff_datetime') > pl.col('tpep_pickup_datetime'))
time_rows = (initial_rows - null_rows - nan_rows - distance_rows - fare_rows) - len(taxi_df)
## removes incorrect dates in data
taxi_df = taxi_df.filter((pl.col('tpep_pickup_datetime').dt.year() == 2024))

year_rows = (initial_rows - null_rows - nan_rows - distance_rows - fare_rows - time_rows) - len(taxi_df)

removed_rows= initial_rows - len(taxi_df)

print("Data Cleaning Summary\n\n")
print(f"Initial row count: {initial_rows}")
print(f"Final row count: {len(taxi_df)}\n")
print(f"Total rows removed {removed_rows}")
print(f"Rows removed due to null/missing values values: {null_rows}")
print(f"Rows removed due to nan values: {nan_rows}")
print(f"Rows remove due to invalid distance: {distance_rows}")
print(f"Rows remove due to unwanted/invlaid fare range: {fare_rows}")
print(f"Rows remove due to invalid pickup/dropoff times: {time_rows}")
print(f"Rows remove due to dates being out of range : {year_rows}")

taxi_df.write_parquet('data/cleaned_data.parquet')



Data Cleaning Summary


Initial row count: 2964624
Final row count: 2869558

Total rows removed 95066
Rows removed due to null/missing values values: 0
Rows removed due to nan values: 0
Rows remove due to invalid distance: 60371
Rows remove due to unwanted/invlaid fare range: 34569
Rows remove due to invalid pickup/dropoff times: 112
Rows remove due to dates being out of range : 14


### Cleaning Lookup CSV

In [268]:
lookup_df = lookup_df.filter(
    (pl.col('Borough') != 'Unknown')&
    (pl.col('Borough') != 'N/A'))
print(lookup_df['Borough'].value_counts())

shape: (6, 2)
┌───────────────┬───────┐
│ Borough       ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ Bronx         ┆ 43    │
│ Brooklyn      ┆ 61    │
│ Manhattan     ┆ 69    │
│ EWR           ┆ 1     │
│ Queens        ┆ 69    │
│ Staten Island ┆ 20    │
└───────────────┴───────┘


## Feature Engineering


In [ ]:
test_df=taxi_df
#Temporal features
taxi_df = taxi_df.with_columns( 
    #Create col that indexes pickup hour
    (pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour')),
    #Create col that indexes weekday, starting at 0
    (pl.col('tpep_pickup_datetime').dt.weekday()-1).alias('pickup_day_of_week')
    ).with_columns(
        # Creates boolean weekend col
        (pl.col('pickup_day_of_week') >= 5).alias('is_weekend')
    )
#Trip Features
taxi_df  = taxi_df.with_columns(
    (pl.col('tpep_dropoff_datetime')- pl.col('tpep_pickup_datetime'))
    #Convert trip duration minutes distance to interger
    .dt.total_minutes().alias('trip_duration_minutes')
    ).with_columns(
        #accounts for zero denominator error
        pl.when(pl.col('trip_duration_minutes') > 0)
        .then(pl.col('trip_distance')* 60/(pl.col('trip_duration_minutes')))
        .otherwise(0) #if duration 0 -> just 0 mph
        .alias('trip_speed_mph')
    )
taxi_df = taxi_df.with_columns(
    pl.col('trip_distance').log1p().alias('log_trip_distance'))
#Fare Features
taxi_df = taxi_df.with_columns(
        pl.when(pl.col('trip_distance') > 0)
        .then(pl.col('fare_amount')/(pl.col('trip_distance')))
        .otherwise(0) #if distance 0 -> just 0
        .alias('fare_per_mile'),
        pl.when(pl.col('trip_duration_minutes') > 0)
        .then(pl.col('fare_amount')/(pl.col('trip_duration_minutes')))
        .otherwise(0) #if distance 0 -> just 0
        .alias('fare_per_minute')
        )
#Zone Features



 Error lookup.csv not found downloading...



ComputeError: invalid utf-8 sequence